# 통합 데이터 전처리 및 마스터셋 구축
> **목적**: 3개 테이블을 병합하고 이용강도(I), 이용빈도(F), 소비습관(H) 분석에 필요한 모든 파생 변수를 통합 산출하여 데이터 정합성 확보

### 주요 작업 내용:
1. **데이터 통합**: 월별 5개 산재 테이블(회원/신용/승인/청구/잔액)을 병합하여 다각적 고객 프로파일 **확보**
2. **데이터 정제**: '_' 및 NaN 결측치 처리, 연령대/Life_Stage/등급 정보의 수치 데이터 **산출**
3. **통합 피처 엔지니어링**: I, F, H 3개 주제의 모든 파생 변수(변화율, 비중, 소비 추세 등)를 누락 없이 **생성**
4. **시계열 라벨링**: 차월(2019.01) 매출 0원 발생 유무를 기준으로 `is_churn` 타겟 변수 **정의**

In [ ]:
# STEP 0. 환경 설정 및 경로 정의

import pandas as pd
import numpy as np
import polars as pl
import os
from functools import reduce
from datetime import datetime

# 데이터 경로 및 설정
base_path = r'C:\Users\ehdnj\OneDrive\Desktop\내일배움캠프 파일\python\최종 프로젝트\AI hub 금융 데이터'
months = ['201807', '201808', '201809', '201810', '201811', '201812']
output_filename = "total_master_dataset.csv"

print(f"STEP 0: 프로세스 시작 ({datetime.now()})")

In [ ]:
# STEP 1. 전처리 함수 (수치화 및 결측치 정제)

def run_final_preprocessing(df):
    # (1) '_' 결측치 및 실제 NaN 0으로 변환
    df = df.replace('_', 0).fillna(0)
    
    # (2) 주요 수치 컬럼 강제 numeric 변환 (dtype object 방지)
    numeric_cols = [
        '이용금액_신용_B0M', '이용금액_신용_B1M', '이용금액_신용_B2M',
        '이용건수_신용_B0M', '이용건수_신용_B1M', '이용건수_신용_B2M',
        '이용금액_온라인_B0M', '이용금액_쇼핑_B0M', '이용금액_유틸리티_B0M', '이용금액_고액_B0M',
        '이용건수_온라인_B0M', '활동일수_B0M', '이용한도_B0M', 'CA한도금액', '청구금액_B0',
        '잔액_B0M', '잔액_일시불_B0M', '잔액_할부_B0M'
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    
    # (3) 연령대 수치화 ('20대' -> 20)
    age_map = {'20대': 20, '30대': 30, '40대': 40, '50대': 50, '60대': 60, '70대이상': 70}
    if '연령' in df.columns:
        df['연령'] = df['연령'].map(age_map).fillna(0).astype(int)
    
    # (4) Life_Stage 수치화 ('1.Single' -> 1)
    if 'Life_Stage' in df.columns:
        df['Life_Stage'] = df['Life_Stage'].astype(str).str.extract('(\d+)').fillna(0).astype(int)
    
    # (5) 등급 코드 숫자 변환
    target_cols = ['VIP등급코드', '최상위카드등급코드']
    for col in target_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)
            
    return df

In [ ]:
# STEP 2. 통합 피처 엔지니어링 (I, F, H 주제별 피처 완전 통합)

def extract_master_features(df):
    print("STEP 2: 주제별 통합 파생변수 생성 중 (Intensity + Frequency + Habit)...")
    eps = 1e-9
    
    # --- [1. 이용강도 (Intensity) 관련 피처] ---
    df['소비기울기'] = df['이용금액_신용_B0M'] - df['이용금액_신용_B1M']
    df['이용금액_변화율'] = df['소비기울기'] / (df['이용금액_신용_B1M'] + eps)
    df['고액결제_비중'] = df['이용금액_고액_B0M'] / (df['이용금액_신용_B0M'] + eps)
    df['온라인_결제비중'] = df['이용금액_온라인_B0M'] / (df['이용금액_신용_B0M'] + eps) # Intensity/Habit 공통
    df['한도사용률'] = df['이용금액_신용_B0M'] / (df['이용한도_B0M'] + eps)
    df['이용금액_3M_Avg'] = df[['이용금액_신용_B0M', '이용금액_신용_B1M', '이용금액_신용_B2M']].mean(axis=1)

    # --- [2. 이용빈도 (Frequency) 관련 피처] ---
    df['이용건수_기울기'] = df['이용건수_신용_B0M'] - df['이용건수_신용_B1M']
    df['이용건수_변화율'] = df['이용건수_기울기'] / (df['이용건수_신용_B1M'] + eps)
    df['활동밀도'] = df['활동일수_B0M'] / 31
    df['일평균_결제건수'] = df['이용건수_신용_B0M'] / (df['활동일수_B0M'] + eps)
    df['건수기준_온라인비중'] = df['이용건수_온라인_B0M'] / (df['이용건수_신용_B0M'] + eps)
    df['이용건수_3M_Avg'] = df[['이용건수_신용_B0M', '이용건수_신용_B1M', '이용건수_신용_B2M']].mean(axis=1)

    # --- [3. 소비습관 (Habit) 관련 피처] ---
    df['온라인_비중'] = df['이용금액_온라인_B0M'] / (df['이용금액_신용_B0M'] + eps) # 위와 동일
    df['쇼핑_비중'] = df['이용금액_쇼핑_B0M'] / (df['이용금액_신용_B0M'] + eps)
    df['유틸리티_비중'] = df['이용금액_유틸리티_B0M'] / (df['이용금액_신용_B0M'] + eps)
    df['Ticket_Size'] = df['이용금액_신용_B0M'] / (df['이용건수_신용_B0M'] + eps)
    df['건당_온라인금액'] = df['이용금액_온라인_B0M'] / (df['이용건수_온라인_B0M'] + eps)
    
    # 패턴 변화 추세 (전달 대비 온라인 비중 변화)
    prev_online_ratio = df['이용금액_온라인_B1M'] / (df['이용금액_신용_B1M'] + eps)
    df['온라인비중_변화량'] = df['온라인_비중'] - prev_online_ratio
    
    return df

In [ ]:
# STEP 3. 월별 데이터 로드 및 병합 (스키마 확장)

def make_monthly_master(month):
    print(f"STEP 3: {month} 데이터 통합 진행 중...")
    key_cols = ['기준년월', '발급회원번호']
    try:
        # F1. 회원정보
        f1_path = os.path.join(base_path, f"01.카드 회원정보\\{month}_회원정보.csv")
        df1 = pd.read_csv(f1_path, usecols=key_cols + ['연령', 'VIP등급코드', 'Life_Stage', '최상위카드등급코드'])

        # F2. 신용정보
        df2 = pd.read_csv(os.path.join(base_path, f"02.카드 신용정보\\{month}_신용정보.csv"), usecols=key_cols + ['CA한도금액', '이용한도_B0M'])
        
        # F3. 승인매출정보 (모든 주제의 원천 컬럼 로드)
        f3_cols = key_cols + [
            '이용금액_신용_B0M', '이용금액_신용_B1M', '이용금액_신용_B2M',
            '이용건수_신용_B0M', '이용건수_신용_B1M', '이용건수_신용_B2M',
            '이용금액_온라인_B0M', '이용금액_온라인_B1M', '이용금액_쇼핑_B0M', '이용금액_유틸리티_B0M', 
            '이용금액_고액_B0M', '이용건수_온라인_B0M', '활동일수_B0M'
        ]
        df3 = pd.read_csv(os.path.join(base_path, f"03.카드 승인매출정보\\{month}_승인매출정보.csv"), usecols=f3_cols)

        # F4. 청구정보
        df4 = pd.read_csv(os.path.join(base_path, f"04.카드 청구정보\\{month}_청구정보.csv"), usecols=key_cols + ['청구금액_B0'])
        
        # F5. 잔액정보
        df5 = pd.read_csv(os.path.join(base_path, f"05.카드 잔액정보\\{month}_잔액정보.csv"), usecols=key_cols + ['잔액_B0M', '잔액_일시불_B0M', '잔액_할부_B0M'])

        # 병합 및 전처리
        dfs = [df1, df2, df3, df4, df5]
        df_merged = reduce(lambda left, right: pd.merge(left, right, on=key_cols, how='left'), dfs)
        
        df_merged = run_final_preprocessing(df_merged)
        return df_merged
    except Exception as e:
        print(f"❌ {month} 로드 실패: {e}")
        return None

In [ ]:
# STEP 4. 최종 마스터셋 구축 (라벨링 및 트렌드 피처 생성)

if __name__ == "__main__":
    import polars as pl
    
    print("🚀 통합 마스터 전처리 프로세스를 시작합니다. (Polars & Parquet 고도화)")
    
    # 1. 월별 데이터 로드 및 1차 병합
    df_list = []
    for m in months:
        temp_df = make_monthly_master(m)
        if temp_df is not None:
            df_list.append(temp_df)
    
    # 전체 기간 통합
    total_df = pd.concat(df_list, ignore_index=True)
    print(f"✅ 기초 데이터 통합 완료: {len(total_df):,}건")

    # 2. 통합 피처 엔지니어링 적용 (I, F, H 모든 피처 산출)
    total_df = extract_master_features(total_df)

    # 3. 시계열 트렌드(T-1) 및 이탈 라벨링(T+1) 생성
    # 앞서 정의했던 all_features 리스트를 기반으로 트렌드 데이터 및 Y값 생성
    all_features = [
        '이용금액_신용_B0M', '이용금액_할부_B0M', '이용금액_CA_B0M', '이용금액_체크_B0M', 
        '이용금액_온라인_B0M', '이용금액_쇼핑_B0M', '이용금액_유틸리티_B0M', '이용금액_고액_B0M',
        '이용건수_신용_B0M', '이용건수_온라인_B0M', '활동일수_B0M'
    ]
    
    print("STEP 4-3: 시계열 관측 기반 라벨링 및 전달(T-1) 데이터 생성 중...")
    
    final_master = create_trend_labeled_df(total_df, all_features)

    # 4. 최종 저장 및 메모리 최적화 (Parquet 전환)
    output_filename = "final_master_dataset.parquet"
    
    final_master = final_master.replace([np.inf, -np.inf], 0).fillna(0)
    final_master = optimize_dtypes(final_master) # 메모리 최적화 함수 호출
    
    # [고도화] CSV 대신 Parquet으로 저장하여 처리 속도 및 용량 최적화 (Polars 활용)
    pl_master = pl.from_pandas(final_master)
    pl_master.write_parquet(output_filename, compression='zstd')
    
    print("\n" + "="*50)
    print(f"🎉 모든 작업 완료! '{output_filename}' (총 {len(final_master):,}건) 생성됨.")
    print("이 파일을 사용하여 Intensity, Frequency, Habit 모델링을 시작하세요.")
    print("="*50)